In [3]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


## Step 1: Imports and Device Setup
- First, fetch the necessary deep learning packages and configure the script to run on your GPU if you have one available

In [1]:
import torch
import torchvision

print("PyTorch Version:", torch.__version__)
print("Torchvision Version:", torchvision.__version__)
print("Is GPU available on Jubail?:", torch.cuda.is_available())

PyTorch Version: 2.2.2+cu118
Torchvision Version: 0.17.2+cu118
Is GPU available on Jubail?: True


In [5]:
# 1. import and device setup
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import tqdm as tqdm

Using device: cuda


In [6]:
# setup device configuration (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Step 2: Data Preprocessing and Loaders
- CIFAR-10 has 10 classes. We will apply random cropping and horizontal flipping as data augmentation to prevent overfitting, followed by standard normalization

In [9]:
# Data augmentation for training, standard normalization for testing
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Download and load CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


## 3. Modifying ResNet-18 for CIFAR-10
- Instead of writing 18 layers from scratch, we can load PyTorch's native resnet17. however, the standard version starts with a 7x7 convolution and max pooling, which reduces a 32 x 32 image down to 4x4 instantly.
- To fix this, swap the first layer to a 3 x 3 convolution, remove the max pool and match the final fully connected layer (fc) to output 10 classes

In [13]:
def get_resnet18_for_cifar10():
  # load standard resnet18 (unpretrained, so we train from scratch)
  model = torchvision.models.resnet18(weights=None)

  # 1. change first conv layer from 7x7 (stride 2) to 3x3 (stride 1)
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

  # 2. remove the MaxPool2d layer (replace with Identity) so we don't lose spatial resolution
  model.maxpool = nn.Identity()

  # 3. change final layer to outptu 10 classes instead of 1000
  model.fc = nn.Linear(512, 10)

  return model.to(device)

model = get_resnet18_for_cifar10()

## 4. Loss Function and Optimizer
- We will use Cross-Entropy Loss and Stochastic Gradient Descent (SGD) with momentum, which historically performs exceptionally well on CIFAR-10

In [19]:
# initial parameters
num_epochs = 200
initial_lr = 0.1

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=0.9, weight_decay=5e-4)

# setup cosine annealing scheduler
# T_max is set to num_epochs so the curve stretches perfectly across the whole training run
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0)

## 5. Training Loop
- This function runs through the dataset, calculates the loss, performs backpropagation, and updates the weights

In [20]:
def train(epoch):
  model.train()
  train_loss = 0
  correct = 0
  total = 0

  # wrap loader in tqdm for a nice progress bar
  progress_bar = tqdm.tqdm(trainloader, desc=f"Epoch {epoch}")
  for batch_idx, (inputs, targets) in enumerate(progress_bar):
    inputs, targets = inputs.to(device), targets.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    _, predicted = outputs.max(1)
    total += targets.size(0)
    correct += predicted.eq(targets).sum().item()

    progress_bar.set_postfix(Loss=f"{train_loss/(batch_idx+1):.3f}", Acc=f"{100.*correct/total:.2f}%") 


## 6. Testing Loop
- After each training epoch, evaluate the model on unseen test images to track performance accuracy

In [21]:
def test():
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    print(f"\n[Test Set] Loss: {test_loss/len(testloader):.3f} | Acc: {100.*correct/total:.2f}%\n")

## 7. Run the Training
- Let's tie it all together and run it for 30 epochs

In [22]:
num_epochs = 200
initial_lr = 0.1

for epoch in range(1, num_epochs + 1):
  # train the model for one epoch
  train(epoch)
  # evaluate on the test set
  test()
  # update the learning rate following the cosine curve
  scheduler.step()

Epoch 1: 100%|██████████| 391/391 [00:04<00:00, 85.56it/s, Acc=79.30%, Loss=0.608]



[Test Set] Loss: 0.752 | Acc: 75.80%



Epoch 2: 100%|██████████| 391/391 [00:04<00:00, 85.53it/s, Acc=84.43%, Loss=0.457]



[Test Set] Loss: 0.736 | Acc: 76.34%



Epoch 3: 100%|██████████| 391/391 [00:04<00:00, 85.47it/s, Acc=85.59%, Loss=0.424]


...


[Test Set] Loss: 0.162 | Acc: 95.59%



Epoch 199: 100%|██████████| 391/391 [00:04<00:00, 86.21it/s, Acc=100.00%, Loss=0.002]



[Test Set] Loss: 0.162 | Acc: 95.73%



Epoch 200: 100%|██████████| 391/391 [00:04<00:00, 85.27it/s, Acc=100.00%, Loss=0.002]



[Test Set] Loss: 0.161 | Acc: 95.74%

